# Approche par machine learning

In [24]:
from pathlib import Path

import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import BallTree
from sklearn.metrics import accuracy_score

In [21]:
import sys
sys.path.append("/home/onyxia/work/gtt/Anthony/ajout_ports.py")

from ajout_ports import associer_ports, distance_cote

In [35]:
nom_fichier = Path("/home/onyxia/work/data/data_indian_ocean.feather")

df = pd.read_feather(nom_fichier)

df.columns

navires_sample = pd.Series(df["mmsi"].unique()).sample(n=50, random_state=42)
df_sample = df[df["mmsi"].isin(navires_sample)].copy()

print(f"Taille du sample : {len(df_sample):,} lignes")
print(f"Navires : {df_sample['mmsi'].nunique()}")

# Échantillon de test
mmsi_sample = (
        pd.Series(df["mmsi"].dropna().unique())
        .sample(n=10, random_state=75)
    )
df_sample = df[df["mmsi"].isin(mmsi_sample)].copy()

    # Pipeline complet
df_ports = associer_ports(df_sample, "upply-seaports.csv", seuil_cote_km=100)

Taille du sample : 1,196,126 lignes
Navires : 50


/home/onyxia/work/gtt/Mathias/ajout_ports.py:55: UserWarning: Geometry is in a geographic CRS. Results from 'distance' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["dist_cote_km"] = gdf.geometry.distance(cotes) * 111


In [36]:
# 3. Merge
df = df.merge(df_ports, on=["voyage number", "mmsi"], how="inner")

In [45]:
df.columns

Index(['imo', 'mmsi', 'name', 'latitude', 'longitude', 'timestamp', 'sog',
       'cog', 'nav status', 'nav status code', 'draft', 'time delta (s)',
       'destination', 'information source', 'checked status', 'load type',
       'voyage number', 'origin->destination', 'at port', 'port stay type',
       'wave period Tp (s)', 'significant wave height Hs (m)',
       'mean wave direction (°)', 'sea surface temperature (°K)',
       'air temperature at 2m (°K)', 'eastward wind velocity (m/s)',
       'northward wind velocity (m/s)',
       'mean wave direction relative to vessel (°)', 'lat_depart',
       'lon_depart', 'dist_depart', 'lat_arrivee', 'lon_arrivee',
       'dist_arrivee', 'port_depart_name', 'port_arrivee_name', 'lat_t-1',
       'lon_t-1', 'lat_t-2', 'lon_t-2'],
      dtype='object')

In [41]:
print("---- DEBUG ----")

print("Unique y_test:", np.unique(y_test))
print("Unique preds:", np.unique(y_pred))

print("\nDistribution vrai ports:")
print(test_df["port_arrivee_name"].value_counts().head())

print("\nExemple décodage:")
for i in range(5):
    print(
        le_target.inverse_transform([y_test.iloc[i]])[0],
        "vs",
        le_target.inverse_transform([y_pred[i]])[0]
    )

---- DEBUG ----
Unique y_test: [ 2  3  6  7  9 14 16 19 20]
Unique preds: [ 1  3  4  6  7  9 12 13 14 15 16 18 19 20]

Distribution vrai ports:
port_arrivee_name
Goudiniweg         2790
Khor al Fakkan     1815
Fa'id              1615
Tanjung Pelepas    1274
Tanjong Bin        1142
Name: count, dtype: int64

Exemple décodage:
Goudiniweg vs Perawang
Goudiniweg vs Fa'id
Goudiniweg vs Perawang
Goudiniweg vs Perawang
Goudiniweg vs Khor al Fakkan


In [47]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import xgboost as xgb

# 4. Features temporelles
df = build_features(df)

# 5. Nettoyage
df = df.dropna(subset=["port_depart_name", "port_arrivee_name"])

# 6. Split AVANT encodage
train_df, test_df = split_by_voyage(df)

# éviter warnings pandas
train_df = train_df.copy()
test_df = test_df.copy()

# 7. Encodage feature : port de départ
le_depart = LabelEncoder()
train_df["port_depart_enc"] = le_depart.fit_transform(train_df["port_depart_name"])

test_df = test_df[test_df["port_depart_name"].isin(le_depart.classes_)]
test_df["port_depart_enc"] = le_depart.transform(test_df["port_depart_name"])

# 8. Encodage target : port d’arrivée
le_target = LabelEncoder()
train_df["target_enc"] = le_target.fit_transform(train_df["port_arrivee_name"])

test_df = test_df[test_df["port_arrivee_name"].isin(le_target.classes_)]
test_df["target_enc"] = le_target.transform(test_df["port_arrivee_name"])

# 9. Features
FEATURES = [
    "latitude", "longitude",
    "port_depart_enc",
    "sog",  # speed over ground
    "cog",  # course over ground
    "dist_depart",
    "dist_arrivee",
    "nav status code",
    "draft"
]

X_train = train_df[FEATURES]
y_train = train_df["target_enc"]

X_test = test_df[FEATURES]
y_test = test_df["target_enc"]

# 10. Modèle XGBoost (probabilités)
model = xgb.XGBClassifier(
    n_estimators=150,
    max_depth=6,
    learning_rate=0.1,
    objective="multi:softprob",
    num_class=len(le_target.classes_),
    tree_method="hist",
    n_jobs=-1
)

model.fit(X_train, y_train)

# 11. Prédictions
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

# 12. Accuracy classique
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f}")

# 13. Top-K prédictions
TOP_K = 3

top_k_indices = np.argsort(y_proba, axis=1)[:, -TOP_K:][:, ::-1]

# convertir en noms de ports
top_k_ports = le_target.inverse_transform(top_k_indices.flatten())
top_k_ports = top_k_ports.reshape(top_k_indices.shape)

# 14. Top-K accuracy
def top_k_accuracy(y_true, top_k_preds):
    correct = 0
    for i in range(len(y_true)):
        if y_true.iloc[i] in top_k_preds[i]:
            correct += 1
    return correct / len(y_true)

top_k_acc = top_k_accuracy(y_test, top_k_indices)
print(f"Top-{TOP_K} Accuracy: {top_k_acc:.4f}")

# 15. Exemple d'affichage
for i in range(min(5, len(X_test))):
    print(f"\nSample {i}")
    vrai_port = le_target.inverse_transform([y_test.iloc[i]])[0]
    print(f"Vrai port: {vrai_port}")
    
    for k in range(TOP_K):
        idx = top_k_indices[i, k]
        port = le_target.inverse_transform([idx])[0]
        proba = y_proba[i, idx]
        print(f"Top {k+1}: {port} ({proba:.3f})")

Accuracy: 0.8001
Top-3 Accuracy: 0.9565

Sample 0
Vrai port: Goudiniweg
Top 1: Goudiniweg (1.000)
Top 2: Chattogram (0.000)
Top 3: Ras Laffan (0.000)

Sample 1
Vrai port: Goudiniweg
Top 1: Goudiniweg (1.000)
Top 2: Chattogram (0.000)
Top 3: Khor al Fakkan (0.000)

Sample 2
Vrai port: Goudiniweg
Top 1: Goudiniweg (1.000)
Top 2: Chattogram (0.000)
Top 3: Khor al Fakkan (0.000)

Sample 3
Vrai port: Goudiniweg
Top 1: Goudiniweg (1.000)
Top 2: Chattogram (0.000)
Top 3: Khor al Fakkan (0.000)

Sample 4
Vrai port: Goudiniweg
Top 1: Goudiniweg (1.000)
Top 2: Chattogram (0.000)
Top 3: Khor al Fakkan (0.000)


## Archives

In [33]:
# 4. Features temporelles
df = build_features(df)

# 5. Nettoyage
df = df.dropna(subset=["port_depart_name", "port_arrivee_name"])

# 6. Encodage
le_depart = LabelEncoder()
le_target = LabelEncoder()

df["port_depart_enc"] = le_depart.fit_transform(df["port_depart_name"])
df["target_enc"] = le_target.fit_transform(df["port_arrivee_name"])

# 7. Features
X = df[[
    "latitude", "longitude",
    "lat_t-1", "lon_t-1",
    "lat_t-2", "lon_t-2",
    "port_depart_enc"
]]

y = df["target_enc"]

# 8. Split par voyage
train_df, test_df = split_by_voyage(df)

X_train = train_df[X.columns]
y_train = train_df["target_enc"]

X_test = test_df[X.columns]
y_test = test_df["target_enc"]


# 9. Modèle XGBoost
model = xgb.XGBClassifier(
    n_estimators=150,
    max_depth=6,
    learning_rate=0.1,
    objective="multi:softmax",
    num_class=len(le_target.classes_),
    tree_method="hist",
    n_jobs=-1
)

model.fit(X_train, y_train)

# 10. Test
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f}")

ValueError: Invalid classes inferred from unique values of `y`.  Expected: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21], got [ 0  1  2  3  5  6  7  8  9 11 12 13 14 15 16 17 18 19 20 21 22 23]

ValueError: Invalid classes inferred from unique values of `y`.  Expected: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21], got [ 0  1  2  3  5  6  7  8  9 11 12 13 14 15 16 17 18 19 20 21 22 23]

# Archive

In [ ]:
import xgboost as xgb

In [8]:
# ------------------------------------------------------------------
# Associer ports (VERSION RAPIDE)
# ------------------------------------------------------------------
# ---------------------------------------------------------------------------
# Fonctions principales
# ---------------------------------------------------------------------------

import geopandas as gpd
import numpy as np
from shapely.geometry import Point
from shapely.strtree import STRtree
import geodatasets

# ⚡ cache global (évite de recalculer à chaque appel)
_COAST_TREE = None
_COAST_GEOMS = None

def _load_coast():
    global _COAST_TREE, _COAST_GEOMS

    if _COAST_TREE is None:
        world = gpd.read_file(geodatasets.get_path("naturalearth.land"))
        world = world.to_crs(epsg=3857)

        _COAST_GEOMS = world.geometry.values
        _COAST_TREE = STRtree(_COAST_GEOMS)

def distance_cote_fast(df: pd.DataFrame) -> pd.DataFrame:
    _load_coast()

    df = df.copy()

    # projection directe
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
        crs="EPSG:4326"
    ).to_crs(epsg=3857)

    points = gdf.geometry.values

    # ⚡ calcul vectorisé avec STRtree
    distances = np.array([
        point.distance(_COAST_GEOMS[_COAST_TREE.nearest(point)])
        for point in points
    ])

    gdf["dist_cote_km"] = distances / 1000

    return gdf.drop(columns="geometry")


_PORTS_GDF = None

def _load_ports(seaports_csv):
    global _PORTS_GDF

    if _PORTS_GDF is None:
        ports_db = pd.read_csv(seaports_csv, sep=";", engine="python")
        ports_db = ports_db[["code", "name", "country_code", "latitude", "longitude"]].dropna()

        _PORTS_GDF = gpd.GeoDataFrame(
            ports_db,
            geometry=gpd.points_from_xy(ports_db["longitude"], ports_db["latitude"]),
            crs="EPSG:4326"
        ).to_crs(epsg=3857)

    return _PORTS_GDF


def associer_ports_fast(df_ais, seaports_csv, seuil_cote_km=100):

    # ⚡ distance rapide
    df_ais = distance_cote_fast(df_ais)

    df_sorted = df_ais.sort_values("timestamp")

    ports = df_sorted.groupby(["voyage number", "mmsi"]).agg(
        lat_depart=("latitude", "first"),
        lon_depart=("longitude", "first"),
        dist_depart=("dist_cote_km", "first"),
        lat_arrivee=("latitude", "last"),
        lon_arrivee=("longitude", "last"),
        dist_arrivee=("dist_cote_km", "last"),
    ).reset_index()

    ports = ports[
        (ports["dist_depart"] < seuil_cote_km)
        & (ports["dist_arrivee"] < seuil_cote_km)
    ]

    gdf_ports = _load_ports(seaports_csv)

    def nearest_join(df, lon_col, lat_col, suffix):
        gdf = gpd.GeoDataFrame(
            df,
            geometry=gpd.points_from_xy(df[lon_col], df[lat_col]),
            crs="EPSG:4326"
        ).to_crs(epsg=3857)

        gdf = gpd.sjoin_nearest(
            gdf, gdf_ports, how="left", distance_col=f"dist_port_{suffix}"
        )

        return gdf[["voyage number", "mmsi", "name"]].rename(
            columns={"name": f"port_{suffix}_name"}
        )

    dep = nearest_join(ports, "lon_depart", "lat_depart", "depart")
    arr = nearest_join(ports, "lon_arrivee", "lat_arrivee", "arrivee")

    ports = ports.merge(dep, on=["voyage number", "mmsi"], how="left")
    ports = ports.merge(arr, on=["voyage number", "mmsi"], how="left")

    return ports


# ------------------------------------------------------------------
# Features temporelles
# ------------------------------------------------------------------
def build_features(df):

    df = df.sort_values(["mmsi", "timestamp"]).copy()

    for lag in [1, 2]:
        df[f"lat_t-{lag}"] = df.groupby("mmsi")["latitude"].shift(lag)
        df[f"lon_t-{lag}"] = df.groupby("mmsi")["longitude"].shift(lag)

    return df.dropna()


# ------------------------------------------------------------------
# Split par voyage (IMPORTANT)
# ------------------------------------------------------------------
def split_by_voyage(df, test_size=0.2):

    voyages = df["voyage number"].drop_duplicates()

    train_v, test_v = train_test_split(
        voyages, test_size=test_size, random_state=42
    )

    train_df = df[df["voyage number"].isin(train_v)]
    test_df = df[df["voyage number"].isin(test_v)]

    return train_df, test_df

In [14]:
df_2.head(10)

,trip_uid,timestamp,latitude,longitude,sog,sin_cog,cos_cog,wave_impact_energy,dtg,target_future_avg_speed
0,7361922_1,2015-04-11 07:46:03,25.036822,55.041943,3.007374,-0.884409,0.466713,0.001787,17.762347,0.015916
1,7361922_1,2015-04-11 08:46:03,25.073490,54.989860,5.658344,-0.707107,0.707107,0.001657,18.454239,0.016551
2,7361922_1,2015-04-11 09:46:03,25.146385,54.916536,6.015336,-0.219571,0.975597,0.002787,20.036776,0.017986
3,7361922_1,2015-04-11 10:46:03,25.246310,54.916329,6.139876,0.057575,0.998341,0.004173,19.017581,0.017087
4,7361922_1,2015-04-11 11:46:03,25.347561,54.932402,6.005020,0.069319,0.997595,0.004878,19.024383,0.017108
5,7361922_1,2015-04-11 12:46:03,25.428468,54.941645,4.746639,0.032468,0.999473,0.005632,20.572855,0.018517
6,7361922_1,2015-04-11 13:46:03,25.498446,54.945350,3.557778,-0.404521,-0.914529,-0.003444,22.865149,0.020599
7,7361922_1,2015-04-11 14:46:03,25.522817,54.933061,1.005293,-0.852267,0.523107,0.005124,24.329038,0.021938
8,7361922_1,2015-04-11 15:46:03,25.532165,54.917831,0.000000,0.458929,-0.888473,-0.005655,25.318386,0.022851
9,7361922_1,2015-04-11 16:46:03,25.532165,54.917977,0.000000,-0.921281,-0.388898,0.000132,25.312492,0.022866


In [19]:
# 1. Charger données
df = pd.read_feather("../../data/02_model_ready.feather")

#prendre 1000 trajets uniquement
mmsi_15 = df.head(10)

# 2. Associer ports (rapide)
ports = associer_ports_fast(df, "upply-seaports.csv")

# 3. Merge
df = df.merge(ports, on=["voyage number", "mmsi"], how="inner")

KeyboardInterrupt: 

## B. Version probabiliste

In [32]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import xgboost as xgb

# 4. Features temporelles
df = build_features(df)

# 5. Nettoyage
df = df.dropna(subset=["port_depart_name", "port_arrivee_name"])

# 6. Split AVANT encodage
train_df, test_df = split_by_voyage(df)

# éviter warnings pandas
train_df = train_df.copy()
test_df = test_df.copy()

# 7. Encodage feature : port de départ
le_depart = LabelEncoder()
train_df["port_depart_enc"] = le_depart.fit_transform(train_df["port_depart_name"])

test_df = test_df[test_df["port_depart_name"].isin(le_depart.classes_)]
test_df["port_depart_enc"] = le_depart.transform(test_df["port_depart_name"])

# 8. Encodage target : port d’arrivée
le_target = LabelEncoder()
train_df["target_enc"] = le_target.fit_transform(train_df["port_arrivee_name"])

test_df = test_df[test_df["port_arrivee_name"].isin(le_target.classes_)]
test_df["target_enc"] = le_target.transform(test_df["port_arrivee_name"])

# 9. Features
FEATURES = [
    "latitude", "longitude",
    "lat_t-1", "lon_t-1",
    "lat_t-2", "lon_t-2",
    "port_depart_enc"
]

X_train = train_df[FEATURES]
y_train = train_df["target_enc"]

X_test = test_df[FEATURES]
y_test = test_df["target_enc"]

# 10. Modèle XGBoost (probabilités)
model = xgb.XGBClassifier(
    n_estimators=150,
    max_depth=6,
    learning_rate=0.1,
    objective="multi:softprob",
    num_class=len(le_target.classes_),
    tree_method="hist",
    n_jobs=-1
)

model.fit(X_train, y_train)

# 11. Prédictions
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

# 12. Accuracy classique
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f}")

# 13. Top-K prédictions
TOP_K = 3

top_k_indices = np.argsort(y_proba, axis=1)[:, -TOP_K:][:, ::-1]

# convertir en noms de ports
top_k_ports = le_target.inverse_transform(top_k_indices.flatten())
top_k_ports = top_k_ports.reshape(top_k_indices.shape)

# 14. Top-K accuracy
def top_k_accuracy(y_true, top_k_preds):
    correct = 0
    for i in range(len(y_true)):
        if y_true.iloc[i] in top_k_preds[i]:
            correct += 1
    return correct / len(y_true)

top_k_acc = top_k_accuracy(y_test, top_k_indices)
print(f"Top-{TOP_K} Accuracy: {top_k_acc:.4f}")

# 15. Exemple d'affichage
for i in range(min(5, len(X_test))):
    print(f"\nSample {i}")
    vrai_port = le_target.inverse_transform([y_test.iloc[i]])[0]
    print(f"Vrai port: {vrai_port}")
    
    for k in range(TOP_K):
        idx = top_k_indices[i, k]
        port = le_target.inverse_transform([idx])[0]
        proba = y_proba[i, idx]
        print(f"Top {k+1}: {port} ({proba:.3f})")

Accuracy: 0.3997
Top-3 Accuracy: 0.7073

Sample 0
Vrai port: Goudiniweg
Top 1: Khor al Fakkan (0.245)
Top 2: Perawang (0.228)
Top 3: Ras Laffan (0.188)

Sample 1
Vrai port: Goudiniweg
Top 1: Khor al Fakkan (0.245)
Top 2: Perawang (0.210)
Top 3: Ras Laffan (0.198)

Sample 2
Vrai port: Goudiniweg
Top 1: Ras Laffan (0.251)
Top 2: Khor al Fakkan (0.212)
Top 3: Perawang (0.211)

Sample 3
Vrai port: Goudiniweg
Top 1: Ras Laffan (0.252)
Top 2: Khor al Fakkan (0.233)
Top 3: Perawang (0.196)

Sample 4
Vrai port: Goudiniweg
Top 1: Ras Laffan (0.241)
Top 2: Khor al Fakkan (0.222)
Top 3: Perawang (0.193)


## A. Version simple

In [ ]:
# 4. Features temporelles
df = build_features(df)

# 5. Nettoyage
df = df.dropna(subset=["port_depart_name", "port_arrivee_name"])

# 6. Split AVANT encodage (IMPORTANT)
train_df, test_df = split_by_voyage(df)

# 7. Encodage des features (port de départ)
le_depart = LabelEncoder()
train_df["port_depart_enc"] = le_depart.fit_transform(train_df["port_depart_name"])

# appliquer au test (sans refit)
test_df = test_df[test_df["port_depart_name"].isin(le_depart.classes_)]
test_df["port_depart_enc"] = le_depart.transform(test_df["port_depart_name"])

# 8. Encodage de la target (port d’arrivée)
le_target = LabelEncoder()
train_df["target_enc"] = le_target.fit_transform(train_df["port_arrivee_name"])

# filtrer test pour garder uniquement les classes connues
test_df = test_df[test_df["port_arrivee_name"].isin(le_target.classes_)]
test_df["target_enc"] = le_target.transform(test_df["port_arrivee_name"])

# 9. Features
FEATURES = [
    "latitude", "longitude",
    "lat_t-1", "lon_t-1",
    "lat_t-2", "lon_t-2",
    "port_depart_enc"
]

X_train = train_df[FEATURES]
y_train = train_df["target_enc"]

X_test = test_df[FEATURES]
y_test = test_df["target_enc"]

# 10. Modèle XGBoost
model = xgb.XGBClassifier(
    n_estimators=150,
    max_depth=6,
    learning_rate=0.1,
    objective="multi:softmax",
    num_class=len(le_target.classes_),
    tree_method="hist",
    n_jobs=-1
)

model.fit(X_train, y_train)

# 11. Test
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f}")

/tmp/ipykernel_377696/3543223337.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df["port_depart_enc"] = le_depart.fit_transform(train_df["port_depart_name"])
/tmp/ipykernel_377696/3543223337.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df["target_enc"] = le_target.fit_transform(train_df["port_arrivee_name"])


Accuracy: 0.4019
